In [9]:
import numpy as np
import matplotlib.pyplot as plt

from weather.config import WeatherConfig
from weather.dataset import build_dataset
from weather.normalization import normalize_global

from mlp.mlp import MLP
from mlp.utils import plot_loss


In [10]:
SEED = 42
np.random.seed(SEED)

experiments = {
    "temperature_regression": {
        "task": "regression",
        "losses": ["mse"],
        # "activation": "gelu",
        "activation": "sigmoid",
        "learning_rate": 0.01,
        "hidden_layers_agg": [64, 32],
        "hidden_layers_flat": [128, 64],
        # "epochs": [400],
        "epochs": [20],
        "optimizer": "momentum",
        "adaptive_lr": False,
        "batching": {
            "minibatch_auto": {
                "batch_size": "auto",
                "shuffle": False,
            },
        },
        "early_stopping": True,
        "patience": 5,
    }
}


In [11]:
base_cfg = WeatherConfig(
    data_dir="../data",
    target="temperature",
    window_size=3,
    skip_day=True,
    normalization="global",
    encode_wind_direction=True,
    max_missing_ratio_per_day=0.3,
)

base_cfg.input_variables = [
    "temperature",
    "humidity",
    # "pressure",
    # "wind_speed",
    # "wind_direction",
]

base_cfg.aggregations = {
    "temperature": ["mean", "min", "max"],
    "humidity": ["mean", "min", "max"],
    # "pressure": ["mean", "min", "max"],
    # "wind_speed": ["mean", "max"],
    # "wind_direction": ["mean"],
}

base_cfg.cities = ["Vancouver", "Portland"]


In [13]:
results = {}

for exp_name, cfg_exp in experiments.items():
    print(f"\n=== EXPERIMENT: {exp_name.upper()} ===")

    for mode in ["aggregate", "flatten"]:
        print(f"\n--- Representation: {mode.upper()} ---")

        cfg = base_cfg
        cfg.window_aggregation = mode

        if mode == "flatten":
            cfg.hours_per_day = 24

        # --- BUILD DATASET ---
        X, Y = build_dataset(cfg, verbose=True)
        X, mu, std = normalize_global(X)

        n_inputs = X.shape[1]
        print(f"Input dimension: {n_inputs}")

        # # --- CHOOSE ARCHITECTURE ---
        # hidden_layers = (
        #     cfg_exp["hidden_layers_agg"]
        #     if mode == "aggregate"
        #     else cfg_exp["hidden_layers_flat"]
        # )
        #
        # for batching_name, batching_cfg in cfg_exp["batching"].items():
        #     print(f"\nBatching: {batching_name}")
        #
        #     for loss_name in cfg_exp["losses"]:
        #         for epoch_count in cfg_exp["epochs"]:
        #             print(
        #                 f"\n>>> Mode={mode}, "
        #                 f"Loss={loss_name}, "
        #                 f"Epochs={epoch_count}"
        #             )
        #
        #             # === MODEL (IDENTYCZNIE JAK MNIST) ===
        #             model = MLP(
        #                 layer_sizes=[n_inputs, *hidden_layers, 1],
        #                 task=cfg_exp["task"],
        #                 activation=cfg_exp["activation"],
        #                 learning_rate=cfg_exp["learning_rate"],
        #                 seed=SEED,
        #                 loss=loss_name,
        #                 optimizer=cfg_exp["optimizer"],
        #                 adaptive_lr=cfg_exp["adaptive_lr"],
        #                 lr_decay=0.999,
        #             )
        #
        #             history, weight_hist, acc_hist = model.fit(
        #                 X,
        #                 Y,
        #                 epochs=epoch_count,
        #                 batch_size=batching_cfg["batch_size"],
        #                 shuffle=batching_cfg["shuffle"],
        #                 verbose=True,
        #                 use_tqdm=False,
        #                 early_stopping=cfg_exp["early_stopping"],
        #                 patience=cfg_exp["patience"],
        #             )
        #
        #             # --- EVALUATION ---
        #             y_pred = model.predict(X).ravel()
        #             y_true = Y.ravel()
        #
        #             mse = np.mean((y_pred - y_true) ** 2)
        #             mae = np.mean(np.abs(y_pred - y_true))
        #             hit = np.mean(np.abs(y_pred - y_true) <= 2.0)
        #
        #             print(f"MSE: {mse:.4f}")
        #             print(f"MAE: {mae:.4f}")
        #             print(f"Hit |err|≤2°C: {hit:.4f}")
        #
        #             results[(mode, epoch_count)] = {
        #                 "mse": mse,
        #                 "mae": mae,
        #                 "hit": hit,
        #             }
        #
        #             plot_loss(
        #                 history,
        #                 title=f"Loss | {mode} | {epoch_count} epochs"
        #             )



=== EXPERIMENT: TEMPERATURE_REGRESSION ===

--- Representation: AGGREGATE ---
=== BUILD DATASET START ===
Data directory: /home/matti/1. Mati/MSI - semestr 2/Sieci neuronowe/Projekt 3 - Weather prediction/data
Target variable: temperature
Aggregation mode: aggregate
Window size (I): 3, Skip day X: True
Max missing ratio per day: 0.3
Input variables: ['temperature', 'humidity']
Loading temperature_train.csv
Loading humidity_train.csv
Number of cities: 2

--- City [1/2]: Vancouver ---


Vancouver | windows: 100%|██████████| 1518/1518 [00:02<00:00, 515.10it/s]



--- City [2/2]: Portland ---


Portland | windows:  13%|█▎        | 201/1518 [00:00<00:02, 504.58it/s]


KeyboardInterrupt: 